# scratchpad_2 rebuilt

This notebook keeps the core experiment from the old `scratchpad_2`: a CUDA-backed square-torus rate simulation with recurrent low-rank weights, diffusion, a Hebbian-style plasticity step, and a live Plotly/ipywidgets display.

Run the setup cells top to bottom. The live simulation cell creates controls only; click **Start** when you want the continuous simulation to run.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import threading
import time

import cupy as cp
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from topologies import square_torus
from weights_cuda import WeightMatrixCUDA

cp.cuda.Device(0).use()
print(f"CuPy {cp.__version__}; CUDA devices: {cp.cuda.runtime.getDeviceCount()}")

## Configuration

Edit this cell for the experiment size and parameters. The default matches the useful recent scratchpad settings: `Z=128`, rank-2 low-rank weights initialized with component value `0.25`, RK4 time step `2e-3`, a center-row sinusoidal drive, current diffusion, and optional plasticity.

In [ ]:
@dataclass
class SimulationConfig:
    # Network/grid shape. Total neuron count is Z * Z.
    Z: int = 128

    # WeightMatrixCUDA stores weights as dot(U[parent], V[child]).
    # With rank=2 and component=0.25, the initial edge weight is 2 * 0.25**2 = 0.125.
    rank: int = 2
    initial_component: float = 0.25

    # Montbrio-Pazo-Roxin style rate/voltage dynamics.
    dt: float = 2e-3
    Delta: float = 1.5
    Eta: float = -4.125
    J: float = 15.0

    # Recurrent current and diffusion.
    current_decay: float = 0.33
    diffusion: float = 0.05

    # Plasticity is implemented in this notebook so it is easy to edit.
    plastic_enabled: bool = True
    plastic_lr: float = 1e-5
    A_p: float = 5e-3
    A_m: float = 1e-3
    tau_p: float = 0.5
    tau_m: float = 0.25
    weight_decay: float = 1e-2
    weight_clip: float = 1e7

    # Center-row external drive. The drive is added to every other neuron in the middle row.
    drive_amplitude: float = 4.0
    drive_period: float = 10.0
    drive_phase: float = 2 / 3
    drive_until: float = 1000.0

    # Numeric guards for live experiments.
    r_clip: tuple[float, float] = (0.0, 100.0)
    v_clip: tuple[float, float] = (-500.0, 500.0)
    i_clip: tuple[float, float] = (-500.0, 500.0)


CONFIG = SimulationConfig()

## CUDA kernels

These are kept in the notebook intentionally. Re-running this cell recompiles the kernels, so you can edit the model or plasticity without changing package files or restarting the kernel.

In [ ]:
rk4_step_kernel = cp.ElementwiseKernel(
    "float32 R, float32 V, float32 I, float32 dt, float32 Delta, float32 Eta, float32 J",
    "float32 R_out, float32 V_out",
    r"""
    const float pi  = 3.14159265358979323846f;
    const float pi2 = pi * pi;

    float k1R = Delta / pi + 2.0f * R * V;
    float k1V = V * V + Eta + J * R + I - pi2 * R * R;

    float R2 = R + 0.5f * dt * k1R;
    float V2 = V + 0.5f * dt * k1V;
    float k2R = Delta / pi + 2.0f * R2 * V2;
    float k2V = V2 * V2 + Eta + J * R2 + I - pi2 * R2 * R2;

    float R3 = R + 0.5f * dt * k2R;
    float V3 = V + 0.5f * dt * k2V;
    float k3R = Delta / pi + 2.0f * R3 * V3;
    float k3V = V3 * V3 + Eta + J * R3 + I - pi2 * R3 * R3;

    float R4 = R + dt * k3R;
    float V4 = V + dt * k3V;
    float k4R = Delta / pi + 2.0f * R4 * V4;
    float k4V = V4 * V4 + Eta + J * R4 + I - pi2 * R4 * R4;

    float fac = dt / 6.0f;
    R_out = R + fac * (k1R + 2.0f * k2R + 2.0f * k3R + k4R);
    V_out = V + fac * (k1V + 2.0f * k2V + 2.0f * k3V + k4V);
    """,
    "scratchpad2_rk4_step_f32",
)


syn_update_kernel = cp.RawKernel(
    r"""
extern "C" __global__
void syn_update(
    const float* __restrict__ R,
    const float* __restrict__ U,
    const float* __restrict__ V,
    const float* __restrict__ I_decayed,
    const int*   __restrict__ children,
    int k,
    int N,
    int child_count,
    float* __restrict__ I_out
){
    int row = blockDim.x * blockIdx.x + threadIdx.x;
    if (row >= N) return;

    float Rp = R[row];
    float Ip = I_decayed[row];
    const float* u = U + (size_t)row * k;
    int base = row * child_count;

    for (int cc = 0; cc < child_count; ++cc) {
        int c = children[base + cc];
        const float* v = V + (size_t)c * k;

        float dot = 0.0f;
        for (int kk = 0; kk < k; ++kk) {
            dot += u[kk] * v[kk];
        }

        atomicAdd(&I_out[c], Rp * dot - 0.5f * Ip);
    }
}
""",
    "syn_update",
)


diffuse_kernel = cp.RawKernel(
    r"""
extern "C" __global__
void diffuse_I(
    const float* __restrict__ I_in,
    float* __restrict__ I_out,
    int Z,
    float Ddt
){
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    int N = Z * Z;
    if (idx >= N) return;

    int i = idx / Z;
    int j = idx % Z;
    int ip = (i + 1) % Z;
    int im = (i + Z - 1) % Z;
    int jp = (j + 1) % Z;
    int jm = (j + Z - 1) % Z;

    float center = I_in[idx];
    float lap = I_in[im * Z + j]
              + I_in[ip * Z + j]
              + I_in[i * Z + jm]
              + I_in[i * Z + jp]
              - 4.0f * center;

    I_out[idx] = center + Ddt * lap;
}
""",
    "diffuse_I",
)


plastic_step_kernel = cp.RawKernel(
    r"""
extern "C" __global__
void plastic_step(
    float* __restrict__ U,
    float* __restrict__ V,
    const int* __restrict__ children,
    const float* __restrict__ R,
    const float* __restrict__ dR,
    int k,
    int N,
    int child_count,
    float lr,
    float A_p,
    float A_m,
    float tau_p,
    float tau_m,
    float weight_decay,
    float weight_clip,
    float l2_reg
){
    int edge = blockDim.x * blockIdx.x + threadIdx.x;
    int edge_count = N * child_count;
    if (edge >= edge_count) return;

    int parent = edge / child_count;
    int child = children[edge];
    float* u = U + (size_t)parent * k;
    float* v = V + (size_t)child * k;

    float w = 0.0f;
    float den_u = l2_reg;
    float den_v = l2_reg;
    for (int kk = 0; kk < k; ++kk) {
        float uk = u[kk];
        float vk = v[kk];
        w += uk * vk;
        den_u += uk * uk;
        den_v += vk * vk;
    }

    float wp = fminf(fmaxf(w, -weight_clip), weight_clip);
    float delta = A_p * R[parent] * (R[child] + tau_p * dR[child])
                - A_m * R[child] * (R[parent] - tau_m * dR[parent])
                - weight_decay * wp * wp * wp;
    float scale = lr * delta;

    for (int kk = 0; kk < k; ++kk) {
        float uk = u[kk];
        float vk = v[kk];
        atomicAdd(&u[kk], scale * vk / den_v);
        atomicAdd(&v[kk], scale * uk / den_u);
    }
}
""",
    "plastic_step",
)

## Simulation object

This wraps state allocation, stepping, diagnostics, and host-frame extraction. It uses the current `WeightMatrixCUDA.neighbors` field, so it no longer depends on the older `weights.network` attribute or the removed `plastic_step` method.

In [ ]:
class ScratchpadSimulation:
    def __init__(self, config: SimulationConfig):
        self.config = config
        self._build_network()
        self.reset_state()

    def _build_network(self):
        cfg = self.config

        def initializer(size):
            return cp.full(size, cfg.initial_component, dtype=cp.float32)

        topology = square_torus(cfg.Z)
        self.weights = WeightMatrixCUDA(
            topology,
            rank=cfg.rank,
            weight_initializer=initializer,
            save_network=False,
            use_k2tree=False,
        )

        self.children = self.weights.neighbors.astype(cp.int32, copy=False)
        self.children_flat = self.children.ravel()
        self.N = self.weights.size
        self.child_count = int(self.children.shape[1])
        self.blocks_nodes = ((self.N + 255) // 256,)
        self.blocks_edges = (((self.N * self.child_count) + 255) // 256,)

    def reset_state(self):
        self.t = 0.0
        self.tick = 0
        self.R = cp.zeros(self.N, dtype=cp.float32)
        self.V = cp.zeros(self.N, dtype=cp.float32)
        self.I = cp.zeros(self.N, dtype=cp.float32)
        self.I_tmp = cp.zeros_like(self.I)
        self.R_prev = cp.empty_like(self.R)
        self.dR = cp.zeros_like(self.R)

    def reset_weights(self):
        self._build_network()
        self.reset_state()

    def apply_external_drive(self):
        cfg = self.config
        if self.t >= cfg.drive_until or cfg.drive_amplitude == 0:
            return

        row_start = (cfg.Z * cfg.Z) // 2
        row_stop = row_start + cfg.Z
        drive = cfg.drive_amplitude * cp.sin(
            cp.float32(cp.pi * (self.t - cfg.drive_phase) / cfg.drive_period)
        )
        self.I[row_start:row_stop:2] += drive

    def update_current(self):
        cfg = self.config
        self.I *= cp.float32(cfg.current_decay)
        I_decayed = self.I.copy()
        syn_update_kernel(
            self.blocks_nodes,
            (256,),
            (
                self.R,
                self.weights.U,
                self.weights.V,
                I_decayed,
                self.children_flat,
                cp.int32(self.weights.U.shape[1]),
                cp.int32(self.N),
                cp.int32(self.child_count),
                self.I,
            ),
        )
        self.apply_external_drive()

    def rk4_step(self):
        cfg = self.config
        self.R[:], self.V[:] = rk4_step_kernel(
            self.R,
            self.V,
            self.I,
            cp.float32(cfg.dt),
            cp.float32(cfg.Delta),
            cp.float32(cfg.Eta),
            cp.float32(cfg.J),
        )
        self.t += cfg.dt

    def plastic_step(self):
        cfg = self.config
        if not cfg.plastic_enabled or cfg.plastic_lr == 0:
            return

        plastic_step_kernel(
            self.blocks_edges,
            (256,),
            (
                self.weights.U,
                self.weights.V,
                self.children_flat,
                self.R,
                self.dR,
                cp.int32(self.weights.U.shape[1]),
                cp.int32(self.N),
                cp.int32(self.child_count),
                cp.float32(cfg.plastic_lr),
                cp.float32(cfg.A_p),
                cp.float32(cfg.A_m),
                cp.float32(cfg.tau_p),
                cp.float32(cfg.tau_m),
                cp.float32(cfg.weight_decay),
                cp.float32(cfg.weight_clip),
                cp.float32(1.0),
            ),
        )

    def diffuse_current(self):
        cfg = self.config
        diffuse_kernel(
            self.blocks_nodes,
            (256,),
            (self.I, self.I_tmp, cp.int32(cfg.Z), cp.float32(cfg.diffusion * cfg.dt)),
        )
        self.I, self.I_tmp = self.I_tmp, self.I

    def step(self):
        cfg = self.config
        self.R_prev[:] = self.R
        self.update_current()
        self.rk4_step()
        self.dR[:] = (self.R - self.R_prev) / cp.float32(cfg.dt)
        self.plastic_step()
        self.diffuse_current()
        cp.clip(self.R, cfg.r_clip[0], cfg.r_clip[1], out=self.R)
        cp.clip(self.V, cfg.v_clip[0], cfg.v_clip[1], out=self.V)
        cp.clip(self.I, cfg.i_clip[0], cfg.i_clip[1], out=self.I)
        self.tick += 1

    def run(self, steps: int):
        start = time.perf_counter()
        for _ in range(int(steps)):
            self.step()
        cp.cuda.Stream.null.synchronize()
        return time.perf_counter() - start

    def frame(self, log_scale: bool = False):
        grid = self.R.reshape(self.config.Z, self.config.Z)
        if log_scale:
            grid = cp.log10(cp.maximum(grid, cp.float32(1e-8)))
        return cp.asnumpy(grid)

    def stats(self):
        finite = bool(cp.all(cp.isfinite(self.R)).get())
        return {
            "tick": self.tick,
            "t": self.t,
            "R_min": float(cp.min(self.R).get()),
            "R_mean": float(cp.mean(self.R).get()),
            "R_max": float(cp.max(self.R).get()),
            "I_min": float(cp.min(self.I).get()),
            "I_max": float(cp.max(self.I).get()),
            "finite_R": finite,
        }


def make_sim(config: SimulationConfig = CONFIG):
    sim = ScratchpadSimulation(config)
    print(
        f"Built {sim.config.Z}x{sim.config.Z} torus "
        f"({sim.N:,} neurons, {sim.child_count} children/neuron)."
    )
    return sim

## Build and smoke test

This bounded run compiles the kernels and verifies that the current code path steps without the old `WeightMatrixCUDA.plastic_step` error.

In [ ]:
sim = make_sim(CONFIG)

elapsed = sim.run(300)
stats = sim.stats()
print(f"Smoke run: {stats['tick']} ticks in {elapsed:.3f}s; t={stats['t']:.3f}")
print(
    "R stats:",
    f"min={stats['R_min']:.6f}",
    f"mean={stats['R_mean']:.6f}",
    f"max={stats['R_max']:.6f}",
    f"finite={stats['finite_R']}",
)

first_child = int(sim.children[0, 0].get())
print(f"Example current edge weight W[0,{first_child}] = {float(sim.weights[0, first_child].get()):.6f}")

## Kernel diagnostic

The recurrent-current kernel is compared against a direct vectorized CuPy reference on the current state. This is useful after editing the kernel.

In [ ]:
def validate_recurrent_current_kernel(sim: ScratchpadSimulation):
    cfg = sim.config
    parents = cp.arange(sim.N, dtype=cp.int32)[:, None]
    children = sim.children

    I0 = sim.I.copy()
    R0 = sim.R.copy()

    I_ref = I0.copy()
    I_ref *= cp.float32(cfg.current_decay)
    W_block = sim.weights[parents, children]
    delta = R0[:, None] * W_block - I_ref[:, None] / cp.float32(2.0)
    cp.add.at(I_ref, children.ravel(), delta.ravel())

    I_ker = I0.copy()
    I_ker *= cp.float32(cfg.current_decay)
    I_decayed = I_ker.copy()
    syn_update_kernel(
        sim.blocks_nodes,
        (256,),
        (
            R0,
            sim.weights.U,
            sim.weights.V,
            I_decayed,
            sim.children_flat,
            cp.int32(sim.weights.U.shape[1]),
            cp.int32(sim.N),
            cp.int32(sim.child_count),
            I_ker,
        ),
    )
    cp.cuda.Stream.null.synchronize()
    return float(cp.max(cp.abs(I_ref - I_ker)).get())

current_diff = validate_recurrent_current_kernel(sim)
print(f"max |I_ref - I_kernel| = {current_diff:.6g}")

## Live simulation

The controller below runs the simulation in a background thread and updates the heatmap every frame. Re-run this cell if you edit the controller code; it will stop any previous live thread first.

In [ ]:
class LiveSimulation:
    def __init__(self, sim: ScratchpadSimulation):
        self.sim = sim
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.RLock()

        self.fig = go.FigureWidget(
            data=[
                go.Heatmap(
                    z=self.sim.frame(),
                    colorscale="Viridis",
                    zmin=0,
                    zmax=1,
                    showscale=False,
                    hoverinfo="skip",
                )
            ]
        )
        self.fig.update_layout(
            width=560,
            height=560,
            margin=dict(l=0, r=0, b=0, t=0),
            xaxis=dict(visible=False, fixedrange=True),
            yaxis=dict(visible=False, fixedrange=True),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)",
            uirevision="scratchpad2-live",
        )

        self.start_button = widgets.Button(description="Start", icon="play", button_style="success")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.step_button = widgets.Button(description="Step 100", icon="step-forward")
        self.reset_button = widgets.Button(description="Reset", icon="refresh")
        self.rebuild_button = widgets.Button(description="Rebuild", icon="wrench")

        self.steps_per_frame = widgets.IntSlider(
            value=50,
            min=1,
            max=2000,
            step=1,
            description="Steps/frame",
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.frame_delay_ms = widgets.IntSlider(
            value=10,
            min=0,
            max=100,
            step=1,
            description="Delay ms",
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.zmax = widgets.FloatSlider(
            value=1.0,
            min=0.05,
            max=10.0,
            step=0.05,
            description="z max",
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.log_scale = widgets.Checkbox(value=False, description="log scale")
        self.plastic_enabled = widgets.Checkbox(
            value=self.sim.config.plastic_enabled,
            description="plasticity",
        )
        self.plastic_lr = widgets.FloatLogSlider(
            value=self.sim.config.plastic_lr,
            base=10,
            min=-7,
            max=-3,
            step=0.25,
            description="plastic lr",
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.status = widgets.HTML(value="idle")

        self.start_button.on_click(lambda _: self.start())
        self.stop_button.on_click(lambda _: self.stop())
        self.step_button.on_click(lambda _: self.step_once(100))
        self.reset_button.on_click(lambda _: self.reset_state())
        self.rebuild_button.on_click(lambda _: self.rebuild())
        self.zmax.observe(self._update_zmax, names="value")

    def _sync_config_from_widgets(self):
        self.sim.config.plastic_enabled = bool(self.plastic_enabled.value)
        self.sim.config.plastic_lr = float(self.plastic_lr.value)

    def _update_zmax(self, change=None):
        with self.fig.batch_update():
            self.fig.data[0].zmax = float(self.zmax.value)

    def _draw(self):
        z = self.sim.frame(log_scale=bool(self.log_scale.value))
        stats = self.sim.stats()
        with self.fig.batch_update():
            self.fig.data[0].z = z
            self.fig.data[0].zmax = float(self.zmax.value)
        self.status.value = (
            f"tick={stats['tick']:,} t={stats['t']:.3f} "
            f"R=[{stats['R_min']:.3g}, {stats['R_mean']:.3g}, {stats['R_max']:.3g}] "
            f"finite={stats['finite_R']}"
        )

    def _loop(self):
        while not self._stop.is_set():
            with self._lock:
                self._sync_config_from_widgets()
                for _ in range(max(1, int(self.steps_per_frame.value))):
                    self.sim.step()
                self._draw()
            delay = max(0, int(self.frame_delay_ms.value)) / 1000.0
            if delay:
                time.sleep(delay)
        self.status.value = "stopped"

    def start(self):
        if self._thread is not None and self._thread.is_alive():
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="scratchpad2-live")
        self._thread.start()
        self.status.value = "running"

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def step_once(self, steps=100):
        self.stop()
        with self._lock:
            self._sync_config_from_widgets()
            for _ in range(int(steps)):
                self.sim.step()
            self._draw()

    def reset_state(self):
        self.stop()
        with self._lock:
            self.sim.reset_state()
            self._draw()

    def rebuild(self):
        self.stop()
        with self._lock:
            self.sim.reset_weights()
            self._draw()

    def display(self):
        controls = widgets.VBox(
            [
                widgets.HBox(
                    [
                        self.start_button,
                        self.stop_button,
                        self.step_button,
                        self.reset_button,
                        self.rebuild_button,
                    ]
                ),
                widgets.HBox([self.steps_per_frame, self.frame_delay_ms, self.zmax]),
                widgets.HBox([self.log_scale, self.plastic_enabled, self.plastic_lr]),
                self.status,
            ]
        )
        display(widgets.VBox([controls, self.fig]))


try:
    live.stop()
except NameError:
    pass

live = LiveSimulation(sim)
live.display()

## Manual helpers

These are short commands that are handy while tuning parameters.

In [ ]:
# Stop a live run from another cell if needed.
# live.stop()

# Run a bounded batch without the UI.
# elapsed = sim.run(10_000)
# print(elapsed, sim.stats())

# Save weights for later experiments.
# sim.weights.save("scratchpad_2_weights.npz")

# Start from a fresh state but keep the current weights.
# sim.reset_state()